# Урок 3.6. Stateful-агенты с LangGraph

**Модуль 3 «RAG: Retrieval-Augmented Generation на практике» · Курс «LLM для Python-разработчиков»**

Наш агент из урока 3.5 — 30 честных строк, и для одного вопроса их достаточно. Но попробуйте спросить его: «Сколько дней отпуска мне положено?», а потом — «А если возьму 12, сколько останется?» Второй вопрос он не поймёт: **история диалога умерла вместе с локальной переменной** `messages`. Сегодня разбираемся, как хранить состояние агента по-взрослому, — на **LangGraph**, самом популярном фреймворке для stateful-агентов, — и честно считаем, что он даёт по сравнению с нашим циклом и во что обходится.

> ⚠️ **Формат урока** — как в 3.4–3.5: без живых прогонов, все ячейки с LLM/Qdrant — guarded. Понадобится `pip install "langgraph>=0.2,<0.4"` (пин в `requirements.txt` не случаен: LangGraph новее требует `langchain-core>=1`, а наш ragas 0.2.15 из урока 3.3 — строго `<1`; это второй заход версионной саги, начавшейся с vertexai). Модель — `qwen2.5:3b` (`ollama pull qwen2.5:3b`), Qdrant — из урока 3.2.

## 🎯 Цели урока

После этого урока вы сможете:

- назвать **три врождённые болезни** самодельного цикла — и что именно LangGraph лечит;
- описать агента как **граф**: состояние с редьюсерами, узлы-функции, обычные и условные рёбра;
- объяснить, почему узел возвращает **дельту состояния**, а не всё состояние — и что будет, если перепутать;
- включить **checkpointer** и получить память диалога через `thread_id` — без единой строчки про хранение истории;
- останавливать граф **перед инструментами** (human-in-the-loop) и продолжать с места остановки;
- аргументированно решать: 30 своих строк или фреймворк.

## План

1. Три болезни нашего цикла
2. Состояние и редьюсеры — главная идея LangGraph
3. Узлы, рёбра и готовые детали: ToolNode, tools_condition
4. Checkpointer: память диалога за одну строчку
5. Что ещё умеет граф: interrupt, стриминг, SQLite
6. Честный баланс: свой цикл против фреймворка

⏱ **Время:** ~80–100 минут.

## 1. Три болезни нашего цикла

Цикл из 3.5 прекрасен прозрачностью, но у него три врождённые проблемы — и все три про **состояние**:

1. **Память живёт в локальной переменной.** `messages` создаётся в начале `run_agent` и умирает с `return`. Диалога нет: каждый вопрос — с чистого листа. (В уроке 1.8 мы уже лечили это для чата — списком `history` в RAM; но это память *одного процесса* и *одного пользователя*.)
2. **Маршрут зашит в код цикла.** Пока маршрут один («думай → действуй»), всё хорошо. Добавьте ветвление («перед пишущим инструментом — спросить человека»), потом второй сценарий, потом откат — и цикл превращается в лапшу из if-ов, которую страшно трогать.
3. **Падение = полная потеря.** Умер процесс на третьем шаге — пропали и найденные документы, и ход рассуждения. Начинать заново.

LangGraph атакует все три, поменяв систему координат: агент — это не цикл, а **граф состояний**:

```text
        ┌──────────────── STATE ────────────────┐
        │  messages: [Human, AI, Tool, AI, ...]  │   <- одно состояние,
        └────────────────────────────────────────┘      его видят все узлы

           START
             │
             ▼
        ┌─────────┐   есть tool_calls?   ┌─────────┐
        │  agent  │ ───────да──────────> │  tools  │
        │ (LLM)   │ <───────────────────  │(исполн.)│
        └─────────┘     результаты        └─────────┘
             │ нет
             ▼
            END
```

Узлы — обычные Python-функции над состоянием. Рёбра — кто за кем. Условное ребро — то самое «если модель просит инструменты». Наш цикл 3.5 — ровно этот граф, только нарисованный неявно, кодом. LangGraph просит нарисовать его явно — и за это даёт снимки состояния после каждого узла (память! возобновление!), готовые детали и точки останова.

In [ ]:
import json
import os
import subprocess
import sys
from importlib.metadata import version as pkg_version
from pathlib import Path

import httpx

sys.path.insert(0, os.getcwd())

QDRANT_URL = os.getenv("QDRANT_URL", "http://127.0.0.1:6333")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = os.getenv("LLM_MODEL", "qwen2.5:3b")
COLLECTION = "vectorika_rag_tfidf"
STATE_PATH = Path("service/tfidf_state.json")


def check(label, probe):
    try:
        detail = probe()
        print(f"[ OK ] {label}: {detail}")
        return True
    except Exception as error:
        print(f"[SKIP] {label}: {type(error).__name__} - зависимые ячейки пропустятся")
        return False


def _langgraph():
    from langchain_ollama import ChatOllama          # noqa: F401
    from langgraph.checkpoint.memory import MemorySaver  # noqa: F401
    from langgraph.graph import StateGraph           # noqa: F401
    return f"v{pkg_version('langgraph')}"


LANGGRAPH_AVAILABLE = check("langgraph + langchain-ollama", _langgraph)
QDRANT_AVAILABLE = check(
    "Qdrant", lambda: "v" + httpx.get(QDRANT_URL, timeout=3).json()["version"])
OLLAMA_AVAILABLE = check(
    "Ollama и модель",
    lambda: MODEL if MODEL in [m["name"] for m in httpx.get(
        f"{OLLAMA_URL}/api/tags", timeout=3).json()["models"]] else 1 / 0)

READY = LANGGRAPH_AVAILABLE and QDRANT_AVAILABLE and OLLAMA_AVAILABLE

if QDRANT_AVAILABLE:
    from qdrant_client import QdrantClient

    from service.embedder import TfidfEmbedder

    if not STATE_PATH.exists():
        subprocess.run([sys.executable, "service/index_kb.py"], check=False)
    emb = TfidfEmbedder.load(STATE_PATH)
    qdrant = QdrantClient(url=QDRANT_URL)

    def retrieve(question: str, top_k: int = 3) -> list[dict]:
        hits = qdrant.query_points(COLLECTION, query=emb.embed_query(question),
                                   limit=top_k).points
        return [{**hit.payload, "score": hit.score} for hit in hits]

## 2. Состояние и редьюсеры — главная идея LangGraph

Если из урока запомнить одно — пусть будет это. Состояние графа — типизированный словарь:

```python
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    #                   ^^^^  ^^^^^^^^^^^^
    #                   тип   РЕДЬЮСЕР - как сливать новое со старым
```

Узел **не мутирует** состояние и **не возвращает его целиком**. Узел возвращает **дельту** — «вот что я добавляю»:

```python
def agent_node(state):
    answer = llm.invoke(state["messages"])   # прочитал всё,
    return {"messages": [answer]}            # вернул ТОЛЬКО новое
```

А редьюсер (`add_messages`) решает, как дельта вливается в состояние: для сообщений — дозаписать в конец (наш `messages.append` из 3.5, но объявленный декларативно). Зачем такая церемония:

- **Снимки.** Раз каждое изменение проходит через одно место, LangGraph может сохранять состояние после каждого узла — отсюда и память, и возобновление, и time travel.
- **Безопасная композиция.** Узлы не знают друг о друге и не дерутся за список: каждый отдаёт свою дельту, слияние — забота редьюсера.

Классическая ошибка новичка — вернуть из узла всю историю (`return {"messages": state["messages"] + [answer]}`): `add_messages` дозапишет её к самой себе, и история задвоится. **Дельту, только дельту.**

Инструменты в LangGraph описываются декоратором `@tool` — и вот приятная рифма с уроком 3.5: **докстринг функции становится description инструмента**, тем самым «описанием-промптом». Стратегия переформулировки переезжает из JSON Schema прямо в докстринг:

In [ ]:
if LANGGRAPH_AVAILABLE:
    import ast
    import operator
    from typing import Annotated, TypedDict

    from langchain_core.tools import tool
    from langgraph.graph.message import add_messages

    class AgentState(TypedDict):
        messages: Annotated[list, add_messages]   # дельты сливаются дозаписью

    @tool
    def search_kb(query: str) -> str:
        """Поиск по внутренней базе знаний компании «Векторика» (отпуска,
        больничные, дежурства, деплой, код-ревью). Возвращает три наиболее
        похожих фрагмента документов. Используй для ЛЮБОГО вопроса о правилах
        компании. Если результат не отвечает на вопрос — вызови поиск ещё раз
        с другой формулировкой (официальные термины: вместо «упал прод» —
        «инцидент», «дежурный»)."""
        if not QDRANT_AVAILABLE:
            return "ошибка: поиск недоступен"
        found = retrieve(str(query), top_k=3)
        if not found or found[0]["score"] < 0.05:
            return "ничего похожего не найдено, попробуй другую формулировку"
        return "\n\n".join(
            f"[{c['doc_id']}] {c['title']} (score {c['score']:.2f}):\n{c['text']}"
            for c in found)

    SAFE_OPS = {ast.Add: operator.add, ast.Sub: operator.sub,
                ast.Mult: operator.mul, ast.Div: operator.truediv,
                ast.USub: operator.neg}

    @tool
    def calc(expression: str) -> str:
        """Калькулятор: числа, скобки и операции + - * /.
        Используй для любой арифметики вместо счёта в уме."""
        def evaluate(node):
            if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
                return node.value
            if isinstance(node, ast.BinOp) and type(node.op) in SAFE_OPS:
                return SAFE_OPS[type(node.op)](evaluate(node.left),
                                               evaluate(node.right))
            if isinstance(node, ast.UnaryOp) and type(node.op) in SAFE_OPS:
                return SAFE_OPS[type(node.op)](evaluate(node.operand))
            raise ValueError("допустимы только числа, + - * / и скобки")

        try:
            return str(round(evaluate(ast.parse(str(expression), mode="eval").body), 6))
        except Exception as error:
            return f"ошибка вычисления: {error}"

    TOOLS = [search_kb, calc]
    print("инструменты готовы; description инструмента =",
          repr(calc.description[:50]) + "...")
else:
    print("[пропущено: langgraph не установлен]")

## 3. Узлы, рёбра и готовые детали

Собираем граф из четырёх деталей — две наши, две из коробки:

1. **`agent_node`** (наш) — один ход модели. Заметьте: system-промпт мы **не храним в состоянии**, а подставляем при вызове — иначе он размножится в истории. `bind_tools` привязывает описания инструментов к модели (то, что в 3.5 делал параметр `tools` в запросе).
2. **`ToolNode`** (из коробки) — узел-исполнитель инструментов. Это дословно наш внутренний цикл из 3.5: пройтись по `tool_calls`, выполнить каждый, ошибки вернуть текстом, результаты — сообщениями `tool`. Приятно узнавать свой код в чужом фреймворке — теперь вы знаете, что у него внутри.
3. **`tools_condition`** (из коробки) — условное ребро: в последнем сообщении есть `tool_calls` → идти в `tools`, нет → `END`. Наш `if not calls: return ...`.
4. **Checkpointer** — о нём весь следующий раздел; пока просто передаём в `compile()`.

In [ ]:
if LANGGRAPH_AVAILABLE:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_ollama import ChatOllama
    from langgraph.checkpoint.memory import MemorySaver
    from langgraph.graph import END, START, StateGraph
    from langgraph.prebuilt.tool_node import ToolNode, tools_condition

    SYSTEM_AGENT = (
        "Ты — ассистент по внутренней базе знаний компании «Векторика».\n"
        "Факты о компании бери ТОЛЬКО из результатов search_kb, не из памяти. "
        "Если поиск не нашёл ответа — переформулируй запрос и попробуй ещё раз. "
        "Арифметику считай только через calc. Если ответа в базе нет — скажи: "
        "«В базе знаний нет ответа на этот вопрос». Отвечай кратко, по-русски."
    )

    llm = ChatOllama(model=MODEL, temperature=0.0, base_url=OLLAMA_URL)
    llm_with_tools = llm.bind_tools(TOOLS)

    def agent_node(state: AgentState) -> dict:
        """Один ход модели. System подставляем на вызов, в состоянии не храним."""
        answer = llm_with_tools.invoke(
            [SystemMessage(content=SYSTEM_AGENT)] + state["messages"])
        return {"messages": [answer]}          # ДЕЛЬТА, не вся история!

    builder = StateGraph(AgentState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(TOOLS))   # наш цикл из 3.5, из коробки
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", tools_condition)  # tools или END
    builder.add_edge("tools", "agent")           # результаты - снова думать

    agent_graph = builder.compile(checkpointer=MemorySaver())
    print(agent_graph.get_graph().draw_mermaid())   # схема графа текстом
else:
    print("[пропущено: langgraph не установлен]")

`draw_mermaid()` печатает схему графа в текстовом формате mermaid — Jupyter и GitHub отрисуют её картинкой; но и глазами видно: `START -> agent`, развилка на `tools`/`END`, петля `tools -> agent`. Ровно рисунок из раздела 1.

Запускаем. Вход — дельта начального состояния (наш вопрос), выход — финальное состояние целиком:

In [ ]:
def show_messages(messages) -> None:
    """Компактный трейс: кто что сказал и какие инструменты просил."""
    for message in messages:
        for call in getattr(message, "tool_calls", None) or []:
            print(f"  ai -> {call['name']}({json.dumps(call['args'], ensure_ascii=False)})")
        if message.content:
            print(f"  {message.type:>5}: {str(message.content)}")


if READY:
    config = {"configurable": {"thread_id": "demo-1"}}   # об этом - раздел 4
    result = agent_graph.invoke(
        {"messages": [HumanMessage(content="Сколько дней отпуска положено в году?")]},
        config)
    show_messages(result["messages"])
    # ожидание: human -> ai с вызовом search_kb -> tool с фрагментами -> ai с ответом
else:
    print("[пропущено: нужны langgraph, Qdrant и Ollama]")

## 4. Checkpointer: память диалога за одну строчку

Теперь — то, ради чего всё затевалось. Мы передали в `compile()` **checkpointer** (`MemorySaver`), и у графа появилось суперсвойство: **после каждого узла состояние сохраняется** — снимком, под ключом `thread_id` из конфига.

Что это меняет:

- **Память диалога — бесплатно.** Следующий `invoke` с тем же `thread_id` начинает не с пустого состояния, а с сохранённого: вся история — в `messages`, дописывается редьюсером. Никакого кода «загрузи историю, добавь, сохрани» — сравните с ручным `history` + `MAX_HISTORY` в llm_cli из урока 1.8.
- **Изоляция пользователей.** `thread_id` — граница диалога: у каждого пользователя (чата, тикета) свой id, и истории не смешиваются. Один общий id на всех — классическая утечка чужих вопросов в контекст.
- **Возобновление.** Снимок есть после *каждого* узла, поэтому упавший (или намеренно остановленный — раздел 5) граф продолжается с места остановки, а не с нуля.

Проверка ниже — трёхходовка, отвечающая на вопрос из начала урока:

1. `thread_id="alice"`: «Сколько дней отпуска положено в году?»
2. `thread_id="alice"`: «А если я возьму 12, сколько останется?» — вопрос **бессмысленный без памяти**: что «взять», откуда «останется»? Ожидание: модель поймёт из истории, что речь про отпуск и 28 дней, и позовёт `calc("28-12")`.
3. `thread_id="bob"`: тот же второй вопрос с чужим id — истории нет, и модель обязана растеряться (переспросить или отказаться). Если и тут ответила «16» — у вас смешались диалоги, бегите смотреть thread_id.

In [ ]:
if READY:
    alice = {"configurable": {"thread_id": "alice"}}

    print("=== ход 1 (alice) ===")
    result = agent_graph.invoke(
        {"messages": [HumanMessage(content="Сколько дней отпуска положено в году?")]},
        alice)
    print("ответ:", result["messages"][-1].content[:90])

    print("\n=== ход 2 (alice, память!) ===")
    result = agent_graph.invoke(
        {"messages": [HumanMessage(content="А если я возьму 12, сколько останется?")]},
        alice)
    show_messages(result["messages"][-3:])   # хвост: ожидаем вызов calc и ответ 16

    print("\n=== контроль: тот же вопрос от bob (памяти нет) ===")
    bob = {"configurable": {"thread_id": "bob"}}
    result = agent_graph.invoke(
        {"messages": [HumanMessage(content="А если я возьму 12, сколько останется?")]},
        bob)
    print("ответ:", result["messages"][-1].content[:120])
else:
    print("[пропущено: нужны langgraph, Qdrant и Ollama]")

In [ ]:
if READY:
    # заглянем в сохранённое состояние алисы: что лежит под thread_id
    snapshot = agent_graph.get_state(alice)
    print(f"сообщений в истории alice: {len(snapshot.values['messages'])}")
    print(f"следующий узел (пусто = граф дошёл до END): {snapshot.next}")
    # checkpointer хранит ВСЮ историю снимков - можно листать назад:
    history = list(agent_graph.get_state_history(alice))
    print(f"снимков состояния: {len(history)} (по одному на каждый шаг графа)")
else:
    print("[пропущено: нужны langgraph, Qdrant и Ollama]")

Обратите внимание на последнюю строчку: checkpointer хранит не только текущее состояние, но и **всю историю снимков** — по одному на каждый пройденный узел. Это открывает «машину времени»: взять старый снимок, поправить и продолжить с него (time travel debugging). Для курса это за скобками, но знать, что оно есть, полезно — при отладке агентов это спасает часы.

## 5. Что ещё умеет граф

Три возможности, ради которых графы и заводят, — конспективно:

**Human-in-the-loop.** `compile(..., interrupt_before=["tools"])` — граф **останавливается перед** узлом инструментов: снимок сохранён, `snapshot.next == ("tools",)`, а в последнем сообщении видно, *что именно* модель хочет вызвать. Человек смотрит, одобряет — и `invoke(None, config)` продолжает с места остановки. Помните дисциплину из 3.5 — «пишущие инструменты требуют подтверждения человеком»? Вот её механизм, из коробки. Разберём в решении 3.

**Стриминг шагов.** `graph.stream(...)` отдаёт события по мере прохождения узлов — пользователь видит «ищу в базе... считаю...» вместо замершего курсора. Тот же UX-принцип, что SSE в уроке 3.2, только на уровне шагов агента.

**Настоящая персистентность.** `MemorySaver` живёт в RAM и умирает с процессом — для урока хватает, для прода нет. Меняется одной строчкой: `SqliteSaver.from_conn_string("checkpoints.db")` (пакет `langgraph-checkpoint-sqlite`) или Postgres-вариант — и диалоги переживают рестарт сервиса. Знакомая история? Это же переезд «индекс в RAM → индекс в Qdrant» из урока 3.2, только для памяти агента.

## 6. Честный баланс: свой цикл против фреймворка

| | наш цикл (3.5) | LangGraph |
|---|---|---|
| строк кода | ~30, все ваши | ~30 своих + фреймворк |
| прозрачность | абсолютная | абстракции; но ToolNode = наш цикл |
| память диалога | руками (history, thread-словарь, обрезка) | checkpointer + thread_id |
| возобновление после падения | нет | снимок после каждого узла |
| human-in-the-loop | городить самим | `interrupt_before` |
| сложные маршруты (ветвления, мультиагент) | лапша из if | узлы и рёбра, композиция |
| зависимости | httpx | langchain-core + langgraph (+версионная дисциплина!) |
| отладка | print в своём коде | трейс состояний, но стектрейсы глубже |

Трезвые выводы:

- **Один агент, один вопрос, без памяти** — фреймворк не нужен: наш цикл проще, быстрее ставится и понятнее в отладке.
- **Диалоговая память, остановки на подтверждение, ветвящиеся маршруты, персистентность** — LangGraph начинает окупаться: всё это придётся писать самим, и ваш вариант будет хуже протестирован.
- **Версии — часть цены.** Мы уже дважды заплатили: ragas потребовал `langchain<1`, из-за этого langgraph зажат в `>=0.2,<0.4`. Фреймворковый стек — это обязательство следить за совместимостью. Пины в requirements + смоук-тест после любого обновления.
- Знание «как внутри» никуда не девается: ToolNode — наш цикл, checkpointer — снимки состояния, tools_condition — наш if. Фреймворк экономит код, а не понимание. Именно поэтому урок 3.5 шёл первым.

## ⚠️ Частые ошибки

| ошибка | чем оборачивается | лекарство |
|---|---|---|
| узел возвращает всю историю вместо дельты | `add_messages` дозаписывает её к себе — история дублируется и пухнет | узел возвращает только НОВЫЕ сообщения |
| узел мутирует `state` напрямую | изменения мимо редьюсера: снимки и слияние ломаются | читать можно, изменять — только возвратом дельты |
| `compile(checkpointer=...)`, а в `invoke` нет `thread_id` | ошибка «checkpointer requires thread_id» | каждый вызов — с `configurable.thread_id` |
| один `thread_id` на всех пользователей | чужие вопросы утекают в контекст диалога | id = граница диалога: пользователь/тикет/сессия |
| `MemorySaver` в проде | рестарт сервиса стирает все диалоги | SqliteSaver / Postgres-checkpointer |
| system-промпт в состоянии | плодится с каждым ходом, ест контекст | подставлять при вызове модели (наш `agent_node`) |
| нет предохранителя от цикла agent↔tools | у LangGraph есть дефолт `recursion_limit=25`, но 25 шагов 3B-модели — это долго и дорого | задать свой: `config={"recursion_limit": 8, ...}` — наша стена `max_steps` |
| `@tool` без докстринга | у инструмента нет description — модель выбирает вслепую (а свежие версии и вовсе падают) | докстринг = описание-промпт, пишется как в 3.5 |
| история растёт бесконечно (checkpointer же помнит всё!) | каждый ход дороже предыдущего; контекст переполняется (урок 1.5 — тихая обрезка!) | обрезка истории узлом (решение 2) — MAX_HISTORY из 1.8 возвращается |
| обновили langchain/langgraph «на свежее» | развал совместимости (наша сага: vertexai, ragas, core<1) | пины версий + смоук после обновления |

## 🏋️ Упражнения

**1. Третий инструмент.** Перенесите `get_document(doc_id)` из решения 3.5 в LangGraph-агента (декоратор `@tool`, докстринг-описание, пересборка графа). Проверьте на вопросе «Перечисли все правила отпуска по пунктам» — зовёт ли модель полный документ после поиска?

**2. Обрезка истории.** Checkpointer помнит всё — значит, длинный диалог будет дорожать с каждым ходом (и однажды упрётся в контекст, урок 1.5). Добавьте в граф узел `trim`, который перед ходом модели удаляет старые ходы, оставляя последние три (подсказки: сообщения удаляются возвратом `RemoveMessage(id=...)` — это тоже дельта!; резать историю можно только по границе хода-`HumanMessage`, иначе осиротевшие tool-сообщения сломают шаблон чата).

**3. Human-in-the-loop.** Скомпилируйте граф с `interrupt_before=["tools"]` и пройдите цикл руками: задать вопрос → убедиться, что граф остановился перед инструментами → посмотреть в снимке, какой вызов запрошен → продолжить `invoke(None, config)`. Зачем такой режим пишущим инструментам?

## 📝 Мини-квиз

<details><summary><b>1. Что делает редьюсер add_messages и что должен вернуть узел?</b></summary>

Редьюсер — правило слияния дельты узла с состоянием; для `messages` — дозапись в конец. Узел возвращает **только новые сообщения** (дельту). Вернуть всю историю — значит дозаписать её к самой себе: сообщения задвоятся, история распухнет. Мутировать состояние напрямую тоже нельзя — изменения пройдут мимо снимков.
</details>

<details><summary><b>2. Откуда у LangGraph-агента «память диалога», если мы не писали ни строчки про хранение истории?</b></summary>

Checkpointer сохраняет состояние после каждого узла под ключом `thread_id`. Следующий `invoke` с тем же id стартует из сохранённого снимка — история уже в `messages`, новый вопрос дозаписывается редьюсером. Смена id = новый чистый диалог; общий id на всех = чужие вопросы в контексте.
</details>

<details><summary><b>3. Чем ToolNode отличается от цикла, который мы писали в 3.5?</b></summary>

Ничем принципиальным — это он и есть, готовой деталью: проходит по tool_calls, исполняет функции, ошибки возвращает моделью текстом, результаты кладёт tool-сообщениями. Мы писали его руками именно затем, чтобы, увидев ToolNode, знать его устройство, а не верить в магию.
</details>

<details><summary><b>4. Почему MemorySaver нельзя в прод и что вместо него?</b></summary>

Он хранит снимки в RAM процесса: рестарт сервиса (деплой, падение — урок 3.2 напомнил, как часто это бывает) стирает все диалоги. В прод — персистентный checkpointer: SQLite (`langgraph-checkpoint-sqlite`) или Postgres. Та же логика, что «индекс в RAM → индекс в Qdrant» из 3.2.
</details>

<details><summary><b>5. Агент ушёл в цикл agent↔tools. Что его остановит и где это настраивается?</b></summary>

`recursion_limit` — лимит шагов графа (по умолчанию 25, при превышении — GraphRecursionError). Для 3B-модели дефолт великоват (25 шагов = долгие секунды и деньги) — задаётся в конфиге вызова: `config={"recursion_limit": 8, "configurable": {...}}`. Это та же стена, что наш `max_steps` из 3.5: контроль числа попыток модели не доверяют.
</details>

<details><summary><b>6. Когда LangGraph — оверкилл?</b></summary>

Когда агент один, маршрут — «думай→действуй», памяти между вызовами не нужно: 30 строк из 3.5 проще, прозрачнее и без версионной дисциплины langchain-стека. Фреймворк окупается на памяти диалога, остановках на подтверждение, ветвящихся маршрутах и персистентности — там, где свой код разрастётся и будет хуже протестирован.
</details>

## Решения

**Решение 1 — третий инструмент.** Всё по рецепту 3.5, только описание живёт в докстринге:

In [ ]:
if LANGGRAPH_AVAILABLE:
    from utils import load_documents

    DOCS_BY_ID = {doc.doc_id: doc for doc in load_documents("data")}

    @tool
    def get_document(doc_id: str) -> str:
        """Полный текст документа базы знаний по его id (id виден в результатах
        search_kb в квадратных скобках, например otpusk). Используй, когда
        найденного фрагмента мало и нужен документ целиком."""
        doc = DOCS_BY_ID.get(str(doc_id).strip())
        if doc is None:
            return f"ошибка: документа {doc_id!r} нет. Есть: " + ", ".join(sorted(DOCS_BY_ID))
        return f"[{doc.doc_id}] {doc.title}\n\n{doc.text}"

    TOOLS_V2 = [search_kb, calc, get_document]

    builder_v2 = StateGraph(AgentState)
    llm_v2 = llm.bind_tools(TOOLS_V2)
    builder_v2.add_node("agent", lambda state: {"messages": [llm_v2.invoke(
        [SystemMessage(content=SYSTEM_AGENT)] + state["messages"])]})
    builder_v2.add_node("tools", ToolNode(TOOLS_V2))
    builder_v2.add_edge(START, "agent")
    builder_v2.add_conditional_edges("agent", tools_condition)
    builder_v2.add_edge("tools", "agent")
    agent_v2 = builder_v2.compile(checkpointer=MemorySaver())
    print("граф v2 собран: search_kb + calc + get_document")

    if READY:
        result = agent_v2.invoke(
            {"messages": [HumanMessage(content=
                "Перечисли все правила, связанные с отпуском, по пунктам")]},
            {"configurable": {"thread_id": "sol-1"}})
        show_messages(result["messages"])
else:
    print("[пропущено: langgraph не установлен]")

**Решение 2 — обрезка истории.** Два тонких места. Первое: удаление — тоже дельта! Узел возвращает `RemoveMessage(id=...)` для каждого лишнего сообщения, а редьюсер `add_messages` понимает их как «убрать из состояния». Второе: резать можно только по границе **хода** (сообщения `human`) — если отрезать историю посреди связки «ai с tool_calls → tool с результатом», шаблон чата сломается: tool-сообщение без родительского вызова.

In [ ]:
if LANGGRAPH_AVAILABLE:
    from langchain_core.messages import RemoveMessage

    MAX_TURNS = 3   # держим последние три хода пользователя

    def trim_node(state: AgentState) -> dict:
        """Удаляет ходы старше MAX_TURNS последних. Ход начинается с human."""
        messages = state["messages"]
        human_positions = [i for i, m in enumerate(messages)
                           if m.type == "human"]
        if len(human_positions) <= MAX_TURNS:
            return {}                                  # дельта «ничего не менять»
        cut = human_positions[-MAX_TURNS]              # граница выживших ходов
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:cut]]}

    builder_v3 = StateGraph(AgentState)
    builder_v3.add_node("trim", trim_node)
    builder_v3.add_node("agent", agent_node)
    builder_v3.add_node("tools", ToolNode(TOOLS))
    builder_v3.add_edge(START, "trim")                 # сначала подстричь,
    builder_v3.add_edge("trim", "agent")               # потом думать
    builder_v3.add_conditional_edges("agent", tools_condition)
    builder_v3.add_edge("tools", "agent")
    agent_v3 = builder_v3.compile(checkpointer=MemorySaver())
    print("граф v3: START -> trim -> agent <-> tools")
    print("проверка: задайте 5+ вопросов в одном thread_id и следите за")
    print("len(get_state(...).values['messages']) - история перестанет расти")
else:
    print("[пропущено: langgraph не установлен]")

**Решение 3 — human-in-the-loop.** Останавливаемся перед каждым исполнением инструментов, смотрим и продолжаем. В реальном сервисе «продолжить» нажимает человек (или не нажимает — и это тоже результат):

In [ ]:
if READY:
    agent_hitl = builder.compile(checkpointer=MemorySaver(),
                                 interrupt_before=["tools"])
    config = {"configurable": {"thread_id": "hitl-demo"}}

    result = agent_hitl.invoke(
        {"messages": [HumanMessage(content="Сколько дней отпуска положено в году?")]},
        config)

    snapshot = agent_hitl.get_state(config)
    print("граф остановлен, следующий узел:", snapshot.next)   # ('tools',)
    pending = snapshot.values["messages"][-1].tool_calls
    print("модель запросила:", [(c["name"], c["args"]) for c in pending])

    # человек посмотрел и одобрил - продолжаем С МЕСТА ОСТАНОВКИ:
    result = agent_hitl.invoke(None, config)      # None = «продолжай со снимка»
    print("после одобрения:", result["messages"][-1].content[:90])

    # Зачем это пишущим инструментам: «создать тикет», «отправить письмо» -
    # действия необратимы, и подтверждение человека между намерением модели
    # и исполнением - главный предохранитель (дисциплина из 3.5, механизм - тут)
else:
    print("[пропущено: нужны langgraph, Qdrant и Ollama]")

## Итоги

- **Три болезни цикла 3.5** — память в локальной переменной, маршрут в спагетти, потеря всего при падении — лечатся одной сменой оптики: агент = **граф состояний**, а не цикл.
- **Главная идея LangGraph — дельты и редьюсеры**: узел возвращает «что добавить», редьюсер сливает. Отсюда снимки после каждого узла — а из снимков растёт всё остальное: память, возобновление, time travel.
- **Checkpointer + thread_id = память диалога за одну строчку** — и граница между пользователями. MemorySaver — для урока, SQLite/Postgres — для прода (та же эволюция, что RAM → Qdrant в 3.2).
- **Готовые детали — это наш код из 3.5**: ToolNode — цикл по tool_calls, tools_condition — наш if, `@tool`-докстринг — то самое «описание = промпт». Фреймворк экономит код, а не понимание.
- **Цена фреймворка — версионная дисциплина**: ragas зажал langchain-core ниже единицы, langchain-core зажал langgraph — пины в requirements и смоук после обновлений обязательны.
- **interrupt_before — предохранитель для пишущих инструментов**: намерение модели отделено от исполнения человеком.

| приём урока | родом из |
|---|---|
| инструменты, описание-промпт, ast-калькулятор | урок 3.5 |
| ретривер как инструмент | уроки 2.4, 3.2 |
| MAX_HISTORY/обрезка истории | урок 1.8 (llm_cli), урок 1.5 (тихая обрезка) |
| «RAM → персистентность» | урок 3.2 (индекс в Qdrant) |
| версионные пины и их причины | урок 3.3 (сага ragas) |

**Что дальше.** Агент умеет искать, считать, помнить диалог и останавливаться на подтверждение. Пора готовить всё хозяйство к продакшену: урок 3.7 — **structlog вместо print** (наши трейсы агента станут структурными логами), **латентность p50/p95** (search_ms/llm_ms из 3.2 дорастают до перцентилей), **санитизация ввода** — потому что пользовательский вопрос, как мы знаем с урока 1.2, бывает и промпт-инъекцией.